### Install packages and download models

In [1]:
!pwd

/teamspace/studios/this_studio


In [2]:
!pip install pickleshare gdown
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!git clone https://github.com/yl4579/StyleTTS2.git
%cd StyleTTS2
!pip install datasets SoundFile munch pydub pyyaml librosa nltk matplotlib accelerate transformers phonemizer einops einops-exts tqdm typing-extensions git+https://github.com/resemble-ai/monotonic_align.git
!sudo apt-get install espeak-ng
!git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LibriTTS
!mv StyleTTS2-LibriTTS/Models .


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
fatal: destination path 'StyleTTS2' already exists and is not an empty directory.


⚡️ Tip	Connect GitHub to Studios: https://lightning.ai/u6510210/home?settings=integrations

/teamspace/studios/this_studio/StyleTTS2
  Cloning https://github.com/resemble-ai/monotonic_align.git to /tmp/pip-req-build-wlse3bcv
  Running command git clone --filter=blob:none --quiet https://github.com/resemble-ai/monotonic_align.git /tmp/pip-req-build-wlse3bcv

  Resolved https://github.com/resemble-ai/monotonic_align.git to commit c6e5e6cb19882164027eb6e35118e841eed9298e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... don

### Download dataset



In [3]:
%cd StyleTTS2
!sudo rm -r ./Data Data.zip

[Errno 2] No such file or directory: 'StyleTTS2'
/teamspace/studios/this_studio/StyleTTS2


In [4]:
!gdown --id 12QR4l9fqLJHHA-EBTtCOGFG2vxGoki1e
!unzip Data.zip

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(


Downloading...
From (original): https://drive.google.com/uc?id=12QR4l9fqLJHHA-EBTtCOGFG2vxGoki1e
From (redirected): https://drive.google.com/uc?id=12QR4l9fqLJHHA-EBTtCOGFG2vxGoki1e&confirm=t&uuid=a67ceff1-0949-4f35-b668-9378c0d3c49f
To: /teamspace/studios/this_studio/StyleTTS2/Data.zip
100%|████████████████████████████████████████| 176M/176M [00:04<00:00, 42.2MB/s]
Archive:  Data.zip
  inflating: Data/OOD_texts.txt      
  inflating: Data/train_list.txt     
  inflating: Data/val_list.txt       
   creating: Data/wavs/
  inflating: Data/wavs/0000.wav      
  inflating: Data/wavs/0001.wav      
  inflating: Data/wavs/0002.wav      
  inflating: Data/wavs/0003.wav      
  inflating: Data/wavs/0004.wav      
  inflating: Data/wavs/0005.wav      
  inflating: Data/wavs/0006.wav      
  inflating: Data/wavs/0007.wav      
  inflating: Data/wavs/0008.wav      
  inflating: Data/wavs/0009.wav      
  inflating: Data/wavs/0010.wav      
  inflating: Data/wavs/0011.wav      
  inflating: Data/w

### Change the finetuning config

Depending on the GPU you got, you may want to change the bacth size, max audio length, epiochs and so on.

In [2]:
%cd StyleTTS2
config_path = "Configs/config_ft.yml"

import yaml

config = yaml.safe_load(open(config_path))

/teamspace/studios/this_studio/StyleTTS2


In [6]:
config["data_params"]["root_path"] = "Data/wavs"
config["batch_size"] = 2  # not enough RAM
config["max_len"] = 630
config["epochs"] = 6
config["loss_params"][
    "joint_epoch"
] = 110  # we do not do SLM adversarial training due to not enough RAM
with open(config_path, "w") as outfile:
    yaml.dump(config, outfile, default_flow_style=True)

In [4]:
# download_checkpoint.py
import os
from huggingface_hub import hf_hub_download

def download_checkpoint(repo_id="nonoJDWAOIDAWKDA/Amelia_reviewed1_ft_StyleTTS2", filename="checkpoint.pth"):
    # Specify the current directory as the cache directory
    cache_dir = os.getcwd()
    
    # Download the checkpoint file to the specified directory
    checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename, cache_dir=cache_dir)
    print(f"Checkpoint downloaded to: {checkpoint_path}")

if __name__ == "__main__":
    download_checkpoint()

checkpoint.pth:   0%|          | 0.00/2.03G [00:00<?, ?B/s]

Checkpoint downloaded to: /teamspace/studios/this_studio/StyleTTS2/models--nonoJDWAOIDAWKDA--Amelia1_ft_StyleTTS2/snapshots/c46d1f2dfc8e1ba982f2519becdd0db9c7f5bdbb/checkpoint.pth


### Start finetuning


In [3]:
!accelerate launch --mixed_precision=fp16 train_finetune_accelerate.py --config_path ./Configs/config_ft.yml | tqdm --desc "Training Progress"

Training Progress: 0it [00:00, ?it/s]

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
config.json: 2.23kB [00:00, 5.85MB/s]
pytorch_model.bin: 100%|██████████████████████| 378M/378M [00:01<00:00, 346MB/s]
bert loaded
Training Progress: 1it [00:09,  9.79s/it]bert_encoder loaded
predictor loaded
decoder loaded
text_encoder loaded
predictor_encoder loaded
style_encoder loaded
diffusion loaded
text_aligner loaded
pitch_extractor loaded
mpd loaded
Training Progress: 11it [00:09,  1.54it/s]msd loaded
wd loaded
BERT AdamW (
Parameter Group 0
    amsgrad: False
    base_momentum: 0.85
    betas: (0.9, 0.99)
    capturable: False
    differentiable: False
    eps: 1e-09
    foreach: None
    fused: None
    initial_lr: 1e-05
    lr: 1e-05
    max_lr: 2

In [ ]:
from huggingface_hub import login, HfApi, create_repo
import os
import json
import torch
from datetime import datetime
import shutil
import glob
import yaml
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


def convert_tensor_to_list(obj):
    """Convert torch tensors to lists recursively"""
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    elif isinstance(obj, dict):
        return {key: convert_tensor_to_list(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_tensor_to_list(item) for item in obj]
    elif isinstance(obj, tuple):
        return tuple(convert_tensor_to_list(item) for item in obj)
    else:
        return obj


# Plot training metrics
def plot_training_metrics(log_file, save_dir):
    metrics = {
        "train_loss": [],
        "val_loss": [],
        "dur_loss": [],
        "F0_loss": [],
        "epochs": [],
    }

    try:
        with open(log_file, "r") as f:
            lines = f.readlines()

        for line in lines:
            if "Validation loss" in line:
                # Extract metrics from validation lines
                parts = line.split(",")
                val_loss = float(parts[0].split(":")[-1])
                dur_loss = float(parts[1].split(":")[-1])
                f0_loss = float(parts[2].split(":")[-1])

                metrics["val_loss"].append(val_loss)
                metrics["dur_loss"].append(dur_loss)
                metrics["F0_loss"].append(f0_loss)
                metrics["epochs"].append(len(metrics["val_loss"]))

        # Create plots
        plt.figure(figsize=(15, 5))

        plt.subplot(131)
        plt.plot(metrics["epochs"], metrics["val_loss"])
        plt.title("Validation Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")

        plt.subplot(132)
        plt.plot(metrics["epochs"], metrics["dur_loss"])
        plt.title("Duration Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")

        plt.subplot(133)
        plt.plot(metrics["epochs"], metrics["F0_loss"])
        plt.title("F0 Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")

        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "training_metrics.png"))
        plt.close()

        return metrics

    except Exception as e:
        print(f"Error plotting metrics: {e}")
        return None


# Login to Hugging Face and setup repository
from google.colab import userdata
try:
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
repo_name = "nonoJDWAOIDAWKDA/Amelia_reviewed1_ft_StyleTTS2"
login(token) if token else login()
api = HfApi()

try:
    create_repo(repo_name, exist_ok=True, token=token, repo_type="model")
except Exception as e:
    print(f"Repository already exists or error creating: {e}")

# Load config and checkpoint
config_path = "Configs/config_ft.yml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
config = convert_tensor_to_list(config)

checkpoint_dir = "Models/LJSpeech"
files = [f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")]
if not files:
    raise ValueError(f"No checkpoint files found in {checkpoint_dir}")
latest_checkpoint = sorted(files, key=lambda x: int(x.split("_")[-1].split(".")[0]))[-1]
checkpoint_path = os.path.join(checkpoint_dir, latest_checkpoint)
print(f"Loading checkpoint: {checkpoint_path}")

checkpoint = torch.load(checkpoint_path, map_location="cpu")

# Prepare files for upload
temp_dir = "temp_model"
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir)

# Create directory structure
os.makedirs(os.path.join(temp_dir, "Utils/ASR"), exist_ok=True)
os.makedirs(os.path.join(temp_dir, "Utils/JDC"), exist_ok=True)
os.makedirs(os.path.join(temp_dir, "Utils/PLBERT"), exist_ok=True)

# Create .gitattributes first
with open(os.path.join(temp_dir, ".gitattributes"), "w") as f:
    f.write("*.pth filter=lfs diff=lfs merge=lfs -text\n")
    f.write("*.t7 filter=lfs diff=lfs merge=lfs -text\n")

# Save model components
print("Saving model components...")
required_components = [
    "bert",
    "bert_encoder",
    "decoder",
    "diffusion",
    "mpd",
    "msd",
    "predictor",
    "predictor_encoder",
    "style_encoder",
    "text_aligner",
    "text_encoder",
    "pitch_extractor",
    "wd",
]

for key in required_components:
    if key in checkpoint["net"]:
        component_path = os.path.join(temp_dir, f"{key}.pth")
        torch.save(checkpoint["net"][key], component_path)
        print(f"Saved {key} to {component_path}")
    else:
        print(f"Warning: {key} not found in checkpoint")

# Save checkpoint
checkpoint_save_path = os.path.join(temp_dir, "checkpoint.pth")
torch.save(checkpoint, checkpoint_save_path)
print(f"Saved full checkpoint to {checkpoint_save_path}")

# Copy utility models and configs
print("Copying utility models and configs...")

# ASR
shutil.copy("Utils/ASR/epoch_00080.pth", os.path.join(temp_dir, "Utils/ASR/"))
shutil.copy("Utils/ASR/config.yml", os.path.join(temp_dir, "Utils/ASR/"))
shutil.copy("Utils/ASR/models.py", os.path.join(temp_dir, "Utils/ASR/"))
shutil.copy("Utils/ASR/layers.py", os.path.join(temp_dir, "Utils/ASR/"))

# JDC (F0)
shutil.copy("Utils/JDC/bst.t7", os.path.join(temp_dir, "Utils/JDC/"))
shutil.copy("Utils/JDC/model.py", os.path.join(temp_dir, "Utils/JDC/"))

# PLBERT
shutil.copy("Utils/PLBERT/step_1000000.t7", os.path.join(temp_dir, "Utils/PLBERT/"))
shutil.copy("Utils/PLBERT/config.yml", os.path.join(temp_dir, "Utils/PLBERT/"))
shutil.copy("Utils/PLBERT/util.py", os.path.join(temp_dir, "Utils/PLBERT/"))

# Copy necessary Python modules
print("Copying Python modules...")
shutil.copy("text_utils.py", os.path.join(temp_dir, "text_utils.py"))
shutil.copy("models.py", os.path.join(temp_dir, "models.py"))
shutil.copy("utils.py", os.path.join(temp_dir, "utils.py"))

# Plot and save training metrics
log_file = os.path.join(checkpoint_dir, "train.log")
metrics = plot_training_metrics(log_file, temp_dir)

# Save configs
# 1. Save minimal config.yml required by inference
inference_config = {
    "model_params": config["model_params"],
    "preprocess_params": config["preprocess_params"],
    "F0_path": "Utils/JDC/bst.t7",
    "ASR_config": "Utils/ASR/config.yml",
    "ASR_path": "Utils/ASR/epoch_00080.pth",
    "PLBERT_dir": "Utils/PLBERT/",
}
with open(os.path.join(temp_dir, "config.yml"), "w") as f:
    yaml.dump(inference_config, f, default_flow_style=False)

# 2. Save detailed config.json
config_save = {
    "model_params": convert_tensor_to_list(config["model_params"]),
    "training_config": {
        "epochs": config["epochs"],
        "batch_size": config["batch_size"],
        "max_len": config["max_len"],
        "optimizer": config["optimizer_params"],
        "loss_params": config["loss_params"],
    },
    "preprocess_params": convert_tensor_to_list(config["preprocess_params"]),
    "data_params": convert_tensor_to_list(config["data_params"]),
    "model_state": {
        "epoch": int(checkpoint.get("epoch", 0)),
        "iterations": int(checkpoint.get("iters", 0)),
        "val_loss": float(checkpoint.get("val_loss", 0.0)),
    },
    "training_metrics": metrics if metrics else {},
}

with open(os.path.join(temp_dir, "config.json"), "w") as f:
    json.dump(config_save, f, indent=2)

# Create model card with training metrics and inference instructions
model_card = f"""---
language: en
tags:
- text-to-speech
- StyleTTS2
- speech-synthesis
license: mit
pipeline_tag: text-to-speech
---

# StyleTTS2 Fine-tuned Model

This model is a fine-tuned version of StyleTTS2, containing all necessary components for inference.

## Model Details
- **Base Model:** StyleTTS2-LibriTTS
- **Architecture:** StyleTTS2
- **Task:** Text-to-Speech
- **Last Checkpoint:** {latest_checkpoint}

## Training Details
- **Total Epochs:** {config['epochs']}
- **Completed Epochs:** {checkpoint.get('epoch', 0)}
- **Total Iterations:** {checkpoint.get('iters', 0)}
- **Batch Size:** {config['batch_size']}
- **Max Length:** {config['max_len']}
- **Learning Rate:** {config['optimizer_params']['lr']}
- **Final Validation Loss:** {checkpoint.get('val_loss', 0.0):.6f}

## Model Components
The repository includes all necessary components for inference:

### Main Model Components:
"""

for key in checkpoint["net"].keys():
    model_card += f"- {key}.pth\n"

model_card += """
### Utility Components:
- ASR (Automatic Speech Recognition)
  - epoch_00080.pth
  - config.yml
  - models.py
  - layers.py
- JDC (F0 Prediction)
  - bst.t7
  - model.py
- PLBERT
  - step_1000000.t7
  - config.yml
  - util.py

### Additional Files:
- text_utils.py: Text preprocessing utilities
- models.py: Model architecture definitions
- utils.py: Utility functions
- config.yml: Model configuration
- config.json: Detailed configuration and training metrics

## Training Metrics
Training metrics visualization is available in training_metrics.png

## Directory Structure
├── Utils/
│ ├── ASR/
│ ├── JDC/
│ └── PLBERT/
├── model_components/
└── configs/

## Usage Instructions
1. Load the model using the provided config.yml
2. Ensure all utility components (ASR, JDC, PLBERT) are in their respective directories
3. Use text_utils.py for text preprocessing
4. Follow the inference example in the StyleTTS2 documentation
"""

with open(os.path.join(temp_dir, "README.md"), "w") as f:
    f.write(model_card)

print("\nPreparing to upload to Hugging Face...")
print(f"Files to be uploaded from {temp_dir}:")
for root, _, files in os.walk(temp_dir):
    for file in files:
        print(f"- {os.path.relpath(os.path.join(root, file), temp_dir)}")

try:
    # Upload all files in a single commit
    api.upload_folder(
        folder_path=temp_dir,
        repo_id=repo_name,
        repo_type="model",
        commit_message=f"Upload StyleTTS2 checkpoint {latest_checkpoint} with all inference components",
        ignore_patterns=["*.pyc", "__pycache__"],
    )
    print("\nAll files uploaded successfully!")

    # Verify upload
    files = api.list_repo_files(repo_id=repo_name)
    print("\nFiles in repository:")
    for file in files:
        print(f"- {file}")

except Exception as e:
    print(f"Error during upload: {str(e)}")
finally:
    print("\nCleaning up...")
    shutil.rmtree(temp_dir)
    print(f"\nModel uploaded to: https://huggingface.co/{repo_name}")

Loading checkpoint: Models/LJSpeech/epoch_2nd_00001.pth
Saving model components...
Saved bert to temp_model/bert.pth
Saved bert_encoder to temp_model/bert_encoder.pth
Saved decoder to temp_model/decoder.pth
Saved diffusion to temp_model/diffusion.pth
Saved mpd to temp_model/mpd.pth
Saved msd to temp_model/msd.pth
Saved predictor to temp_model/predictor.pth
Saved predictor_encoder to temp_model/predictor_encoder.pth
Saved style_encoder to temp_model/style_encoder.pth
Saved text_aligner to temp_model/text_aligner.pth
Saved text_encoder to temp_model/text_encoder.pth
Saved pitch_extractor to temp_model/pitch_extractor.pth
Saved wd to temp_model/wd.pth
Saved full checkpoint to temp_model/checkpoint.pth
Copying utility models and configs...
Copying Python modules...

Preparing to upload to Hugging Face...
Files to be uploaded from temp_model:
- pitch_extractor.pth
- training_metrics.png
- style_encoder.pth
- bert_encoder.pth
- config.json
- bert.pth
- .gitattributes
- wd.pth
- decoder.pth
-

bst.t7:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

bert.pth:   0%|          | 0.00/25.2M [00:00<?, ?B/s]

step_1000000.t7:   0%|          | 0.00/25.2M [00:00<?, ?B/s]

epoch_00080.pth:   0%|          | 0.00/94.6M [00:00<?, ?B/s]

...

style_encoder.pth:   0%|          | 0.00/55.5M [00:00<?, ?B/s]

text_aligner.pth:   0%|          | 0.00/31.5M [00:00<?, ?B/s]

text_encoder.pth:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

wd.pth:   0%|          | 0.00/4.70M [00:00<?, ?B/s]


All files uploaded successfully!

Files in repository:
- .gitattributes
- README.md
- Utils/ASR/config.yml
- Utils/ASR/epoch_00080.pth
- Utils/ASR/layers.py
- Utils/ASR/models.py
- Utils/JDC/bst.t7
- Utils/JDC/model.py
- Utils/PLBERT/config.yml
- Utils/PLBERT/step_1000000.t7
- Utils/PLBERT/util.py
- bert.pth
- bert_encoder.pth
- checkpoint.pth
- config.json
- config.yml
- decoder.pth
- diffusion.pth
- models.py
- mpd.pth
- msd.pth
- pitch_extractor.pth
- predictor.pth
- predictor_encoder.pth
- style_encoder.pth
- text_aligner.pth
- text_encoder.pth
- text_utils.py
- training_metrics.png
- utils.py
- wd.pth

Cleaning up...

Model uploaded to: https://huggingface.co/nonoJDWAOIDAWKDA/Amelia_reviewed1_ft_StyleTTS2
